# V18 – Kryptografie Teil 2: Python-Recap

## 🎯 Lernziele
Nach diesem Notebook kannst du …
- den Unterschied zwischen **`str`** (Text) und **`bytes`** (Rohdaten) erklären und zwischen beiden Formaten mit `.encode()` und `.decode()` wechseln,
- Strings mit `.lower()` und `.strip()` normalisieren und mit `==` vergleichen,
- den `in`-Operator nutzen, um zu prüfen, ob ein Element in einer Liste vorkommt (Grundlage für den Wörterbuch-Angriff später),
- eine Datei im **Binärmodus** (`"rb"`) mit `with open(...)` einlesen.

## ⏱️ Zeitbudget
Ca. 15 Minuten als Warm-up vor der eigentlichen Theorie.

## 🧭 TL;DR
Hash-Funktionen arbeiten nicht auf `str`-Objekten, sondern auf rohen **Bytes**. Deshalb müssen wir jeden Text vor dem Hashen per `.encode("utf-8")` in eine Byte-Folge umwandeln. Dateien lesen wir im Binärmodus ein, damit auch Firmware-Dateien korrekt verarbeitet werden.

## 📦 Voraussetzungen
- Python-Grundlagen aus V01–V10 (Variablen, Strings, Listen, Schleifen, Funktionen).
- Du kannst Code-Zellen mit `Shift+Enter` ausführen.

## 1. `bytes` versus `str` – zwei verschiedene Welten

In Python gibt es zwei grundlegend verschiedene Typen, um Zeichenketten zu repräsentieren. Ein **`str`** ist eine Folge von Unicode-Zeichen und damit gut geeignet, um Text lesbar zu halten. Ein **`bytes`**-Objekt hingegen ist eine Folge von ganzen Zahlen zwischen 0 und 255 und repräsentiert rohe Binärdaten, wie sie auf Festplatten, im Netzwerk oder eben in kryptografischen Funktionen vorkommen.

Kryptografische Hash-Funktionen arbeiten ausschließlich auf **Bytes**, weil sie auf der Bit-Ebene rechnen und sich nicht um Textkodierungen kümmern wollen. Deshalb musst du jeden String vor dem Hashen umwandeln.

In [1]:
text = "Hallo"          # str-Objekt
rohdaten = b"Hallo"      # bytes-Objekt (mit b-Präfix)

print(type(text), text)
print(type(rohdaten), rohdaten)
print("Gleich?", text == rohdaten)  # False – verschiedene Typen

<class 'str'> Hallo
<class 'bytes'> b'Hallo'
Gleich? False


### Umwandlung mit `.encode()` und `.decode()`

Die Methode `.encode("utf-8")` auf einem String erzeugt aus ihm ein `bytes`-Objekt. Die Gegenrichtung übernimmt `.decode("utf-8")` auf einem `bytes`-Objekt. `"utf-8"` ist die heute übliche Zeichenkodierung, die alle Unicode-Zeichen darstellen kann. Für reine ASCII-Buchstaben (a–z, A–Z, Ziffern) sieht das Ergebnis gleich aus, lediglich das `b`-Präfix verrät den anderen Typ.

In [2]:
text = "abc"
als_bytes = text.encode("utf-8")
print(als_bytes)              # b'abc'
print(type(als_bytes))        # <class 'bytes'>

zurueck = als_bytes.decode("utf-8")
print(zurueck)                # 'abc'
print(type(zurueck))          # <class 'str'>

b'abc'
<class 'bytes'>
abc
<class 'str'>


Bei Sonderzeichen wie Umlauten wird der Unterschied deutlich sichtbar. Ein einzelner Umlaut belegt in UTF-8 zwei Bytes, während er als `str` ein einzelnes Zeichen bleibt. Diese Eigenheit ist selten ein Problem, es ist aber gut, sie einmal gesehen zu haben.

In [3]:
umlaut = "ö"
print("Länge als str:  ", len(umlaut))            # 1
print("Länge als bytes:", len(umlaut.encode()))   # 2

Länge als str:   1
Länge als bytes: 2


## 2. String-Operationen für Passwort-Checks

Bevor du ein Passwort hashst, lohnt es sich oft, die Eingabe zu **normalisieren**. Typische Schritte sind das Entfernen führender und abschließender Leerzeichen mit `.strip()` sowie die Umwandlung in Kleinbuchstaben mit `.lower()`. Strings vergleichst du anschließend mit `==`, was einen booleschen Wert (`True` oder `False`) liefert.

In [4]:
eingabe = "  Admin  "
normalisiert = eingabe.strip().lower()
print(repr(normalisiert))                  # 'admin'
print(normalisiert == "admin")             # True
print("HALLO".lower() == "hallo")          # True

'admin'
True
True


## 3. Der `in`-Operator für Listen

Ein **Wörterbuch-Angriff** läuft eine Liste häufiger Passwörter durch und prüft, ob eines davon passt. Um in Python zu prüfen, ob ein Element in einer Liste enthalten ist, nutzt du den `in`-Operator. Er liefert `True`, falls das Element gefunden wurde, sonst `False`. Damit lässt sich auch sehr kompakt eine Wortliste durchsuchen.

In [5]:
wortliste = ["test", "1234", "admin", "passwort"]
print("admin" in wortliste)     # True
print("geheim" in wortliste)    # False

# Gleiche Idee als Schleife – so baust du gleich den Angriff
for wort in wortliste:
    if wort == "admin":
        print("gefunden:", wort)
        break

True
False
gefunden: admin


## 4. Dateien im Binärmodus einlesen

Eine Firmware-Datei ist keine Textdatei, sondern eine beliebige Folge von Bytes. Um sie korrekt zu öffnen, nutzt du den Modus `"rb"` (**r**ead **b**inary). Die Methode `.read()` liefert dann ein `bytes`-Objekt, das du direkt an eine Hash-Funktion übergeben kannst. Der `with`-Block sorgt automatisch dafür, dass die Datei nach dem Lesen wieder geschlossen wird, selbst wenn zwischendrin ein Fehler auftritt.

In [6]:
# Wir lesen die Firmware-Datei nur an, um die Länge zu prüfen.
with open("testdaten/firmware_v2.3.0.bin", "rb") as f:
    daten = f.read()

print("Typ:  ", type(daten))
print("Länge:", len(daten), "Bytes")
print("Erste 16 Bytes:", daten[:16])

Typ:   <class 'bytes'>
Länge: 1546 Bytes
Erste 16 Bytes: b'FIRMWARE_HEADER_'


## 5. Mini-Übung – `.encode()` richtig anwenden

Baue den String `"passwort123"` in ein `bytes`-Objekt um und speichere das Ergebnis in der Variablen `als_bytes`. Nutze `.encode("utf-8")`.

In [7]:
# TODO: Erzeuge ein bytes-Objekt aus dem String "passwort123"
als_bytes = b""  # hier anpassen

In [8]:
# ▶️ Selbstkontrolle – einfach ausführen
try:
    assert isinstance(als_bytes, bytes), f"als_bytes muss bytes sein, ist aber {type(als_bytes).__name__}"
    assert als_bytes == b"passwort123", f"Erwartet b'passwort123', bekommen: {als_bytes!r}"
    print("✅ Richtig! Du hast das String in Bytes umgewandelt.")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ Die Variable `als_bytes` existiert noch nicht. Hast du den TODO gelöst?")

❌ Noch nicht ganz: Erwartet b'passwort123', bekommen: b''


## 6. Mini-Übung – Listen-Lookup

Gegeben ist die Wortliste `passwoerter = ["1234", "qwerty", "admin", "hallo"]`. Weise der Variablen `ist_enthalten` das Ergebnis der Prüfung zu, ob der String `"admin"` in der Liste vorkommt.

In [9]:
passwoerter = ["1234", "qwerty", "admin", "hallo"]

# TODO: Prüfe mit dem in-Operator, ob "admin" in der Liste steht
ist_enthalten = False  # hier anpassen

In [10]:
# ▶️ Selbstkontrolle – einfach ausführen
try:
    assert ist_enthalten is True, f"Erwartet True, bekommen: {ist_enthalten}"
    print("✅ Richtig! Der `in`-Operator liefert True.")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ Die Variable `ist_enthalten` existiert noch nicht. Hast du den TODO gelöst?")

❌ Noch nicht ganz: Erwartet True, bekommen: False


## ✅ Zusammenfassung

| Konzept | Bedeutung |
|---|---|
| `"abc".encode("utf-8")` | `str` → `bytes` |
| `b"abc".decode("utf-8")` | `bytes` → `str` |
| `text.strip().lower()` | Eingabe normalisieren |
| `"admin" in liste` | Listen-Lookup, `True`/`False` |
| `with open(pfad, "rb") as f` | Datei im Binärmodus öffnen |

## ➡️ Nächster Schritt
Weiter mit [01_theorie.ipynb](01_theorie.ipynb) – dort lernst du, was eine Hash-Funktion eigentlich ist und warum sie so wichtig für die IT-Sicherheit ist.